<a href="https://colab.research.google.com/github/Olayimka/Stock-market-forecasting/blob/main/stockforecast.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import pandas as pd
import datetime as dt
from datetime import date
import matplotlib.pyplot as plt
import yfinance as yf
import numpy as np
import tensorflow as tf

START = "1999-02-01"
TODAY = date.today().strftime("%Y-%m-%d")
# Define a function to load the dataset
def load_data(ticker):
    data = yf.download(ticker, START, TODAY)
    data.reset_index(inplace=True)
    return data

from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler(feature_range=(0,1))



In [8]:
import numpy as np
import pandas as pd
from tensorflow.keras.layers import Dense, Dropout, LSTM
from tensorflow.keras.models import Sequential

# Define a ticker for demonstration purposes
ticker = 'AAPL' # You can change this to any stock ticker, e.g., 'MSFT', 'GOOG'

# Load the data using the function from the first cell
# This assumes load_data and START, TODAY variables are available from prior execution
data = load_data(ticker)

# Create training and testing DataFrames as per original structure
train = pd.DataFrame(data[0:int(len(data)*0.70)])
test = pd.DataFrame(data[int(len(data)*0.70): int(len(data))])
print(train.shape)
print(test.shape)

# Prepare training data for the model (scaling 'Close' prices)
# 'scaler' is assumed to be defined in the previous cell
data_training_array = scaler.fit_transform(train[['Close']])

# Create the training data set (x_train, y_train)
x_train = []
y_train = []
look_back = 100 # Number of previous days to use for prediction

# Ensure look_back period (100) is not greater than data available
if data_training_array.shape[0] < look_back:
    print(f"Warning: Training data length ({data_training_array.shape[0]}) is less than look-back period ({look_back}). Adjusting look-back.")
    look_back = data_training_array.shape[0] # Adjust look_back if data is too short

for i in range(look_back, data_training_array.shape[0]):
    x_train.append(data_training_array[i-look_back: i, 0])
    y_train.append(data_training_array[i, 0])

# Convert x_train and y_train to numpy arrays
x_train, y_train = np.array(x_train), np.array(y_train)

# Reshape the data for LSTM input: (samples, time_steps, features)
# Here, samples = number of training examples
# time_steps = look_back (100 days)
# features = 1 (Close price)
x_train = np.reshape(x_train, (x_train.shape[0], x_train.shape[1], 1))

# Define the LSTM model architecture
model = Sequential()
model.add(LSTM(units = 50, activation = 'relu', return_sequences=True, input_shape = (x_train.shape[1], 1)))
model.add(Dropout(0.2))

model.add(LSTM(units = 60, activation = 'relu', return_sequences=True))
model.add(Dropout(0.3))

model.add(LSTM(units = 80, activation = 'relu', return_sequences=True))
model.add(Dropout(0.4))

model.add(LSTM(units = 120, activation = 'relu'))
model.add(Dropout(0.5))
model.add(Dense(units = 1))

/tmp/ipykernel_46415/3803036480.py:13: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, START, TODAY)
[*********************100%***********************]  1 of 1 completed
/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


(4778, 6)
(2049, 6)


In [ ]:
import tensorflow as tf
from sklearn.metrics import mean_absolute_error
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

model.compile(optimizer = 'adam', loss = 'mean_squared_error',
metrics=[tf.keras.metrics.MeanAbsoluteError()])
model.fit(x_train, y_train, epochs = 100)

# --- Prepare test data for prediction ---
# Get the last 'look_back' days from the training data
past_100_days = train['Close'].tail(look_back).values

# Concatenate with the test data 'Close' prices
# This combined data is what we'll use to create the sequences for x_test
test_data_combined = np.concatenate((past_100_days, test['Close'].values))

# Reshape for scaling (MinMaxScaler expects 2D array)
test_data_combined_reshaped = test_data_combined.reshape(-1, 1)

# Scale this combined data using the *already fitted* scaler
input_data_scaled = scaler.transform(test_data_combined_reshaped)

x_test = []
# Create x_test using sliding windows of 'look_back'
for i in range(look_back, input_data_scaled.shape[0]):
    x_test.append(input_data_scaled[i-look_back:i, 0])

x_test = np.array(x_test)

# Reshape x_test for LSTM input: (samples, time_steps, features)
x_test = np.reshape(x_test, (x_test.shape[0], x_test.shape[1], 1))

# --- Make predictions ---
y_pred_scaled = model.predict(x_test)

# --- Inverse transform the predictions to get actual price values ---
y_pred_unscaled = scaler.inverse_transform(y_pred_scaled)

# --- Get the actual 'Close' prices from the test set for comparison and plotting ---
y_test_actual = test['Close'].values

# --- Calculate Mean Absolute Error ---
mae = mean_absolute_error(y_test_actual, y_pred_unscaled)
mae_percentage = (mae / np.mean(y_test_actual)) * 100
print("Mean absolute error on test set: {:.2f}%".format(mae_percentage))


# --- Plotting ---
plt.figure(figsize = (12,6))
plt.plot(y_test_actual, 'b', label = "Original Price") # Plot actual unscaled prices in blue
plt.plot(y_pred_unscaled, 'r', label = "Predicted Price") # Plot predicted unscaled prices in red
plt.xlabel('Time')
plt.ylabel("Price")
plt.legend()
plt.grid(True)
plt.show()

Epoch 1/100
147/147 ━━━━━━━━━━━━━━━━━━━━ 44s 249ms/step - loss: 0.0138 - mean_absolute_error: 0.0676
Epoch 2/100
147/147 ━━━━━━━━━━━━━━━━━━━━ 37s 253ms/step - loss: 0.0036 - mean_absolute_error: 0.0360
Epoch 3/100
147/147 ━━━━━━━━━━━━━━━━━━━━ 40s 249ms/step - loss: 0.0034 - mean_absolute_error: 0.0355
Epoch 4/100
147/147 ━━━━━━━━━━━━━━━━━━━━ 37s 250ms/step - loss: 0.0033 - mean_absolute_error: 0.0347
Epoch 5/100
147/147 ━━━━━━━━━━━━━━━━━━━━ 36s 244ms/step - loss: 0.0026 - mean_absolute_error: 0.0318
Epoch 6/100
147/147 ━━━━━━━━━━━━━━━━━━━━ 41s 247ms/step - loss: 0.0027 - mean_absolute_error: 0.0320
Epoch 7/100
147/147 ━━━━━━━━━━━━━━━━━━━━ 41s 248ms/step - loss: 0.0025 - mean_absolute_error: 0.0314
Epoch 8/100
147/147 ━━━━━━━━━━━━━━━━━━━━ 41s 248ms/step - loss: 0.0024 - mean_absolute_error: 0.0309
Epoch 9/100
147/147 ━━━━━━━━━━━━━━━━━━━━ 37s 250ms/step - loss: 0.0022 - mean_absolute_error: 0.0304
Epoch 10/100
147/147 ━━━━━━━━━━━━━━━━━━━━ 37s 250ms/step - loss: 0.0024 - mean_absolute_err